In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')


# PUBLICATION QUALITY SETTINGS


plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'



output_dir = 'publication_figures'
os.makedirs(output_dir, exist_ok=True)

print("=" * 80)
print("REGULATOR FAMILY ANALYSIS ACROSS ANTIBIOTIC TREATMENTS")
print("=" * 80)


# 1. FINAL REGULATOR CLASSIFICATION

regulator_classification = {

    'FrmR':                                    ['rcnR'],
    'LysR':                                    ['allS', 'benM', 'catM', 'cbl', 'cmpR',
                                                'cynR', 'dmlR', 'gbpR', 'gcvA', 'gltC',
                                                'gltR', 'hcaR', 'hdfR', 'iciA', 'metR',
                                                'nac', 'oxyR', 'pcpR', 'perR', 'pgrR',
                                                'syrM1', 'yahB', 'yjiE'],
    'AcrR':                                    ['AcrR', 'acnR', 'acrR', 'adeN', 'betI',
                                                'comR', 'fabR', 'fadR', 'kstR2', 'mcbR',
                                                'nemR', 'qacR', 'rutR', 'slmA', 'tetC',
                                                'ydeS', 'yhgD', 'yxaF'],
    'FadR':                                    ['lldR', 'pdhR'],
    'GntR':                                    ['GntR', 'gntR', 'paaX', 'phnR', 'ydfH',
                                                'yvoA'],
    'Lrp':                                     ['decR', 'lrp', 'lrpC'],
    'LuxR':                                    ['dauR','anoR'],
    'bolA':                                    ['bolA'],
    'PadR':                                    ['yqjI'],
    'Fur':                                     ['fur', 'zur'],
    'AraC':                                    ['VVA1249', 'ada', 'appY', 'araC', 'feaR',
                                                'gadX', 'nimR', 'nphR', 'oruR', 'rhaS',
                                                'rrp12', 'virS', 'xylS'],
    'MerR':                                    ['cbpM', 'cueR', 'hmrR', 'soxR', 'zntR'],
    'MocR':                                    ['gabR', 'norG'],
    'P(II) protein':                           ['glnK'],
    'NrdR':                                    ['nrdR'],
    'TCS':                                     ['atoC', 'baeR', 'btsR', 'envZ', 'exaE',
                                                'gacA', 'glnG', 'glnL', 'kdpD', 'kdpE',
                                                'ompR', 'pdtaR', 'phoB', 'phoP', 'phoR',
                                                'phoU', 'pleD', 'pmrA', 'pmrB', 'qseB',
                                                'qseC', 'rcsC', 'rstA', 'sasA', 'srrA'],
    'DksA':                                    ['dksA'],
    'H-NS':                                    ['hvrA'],
    'IclR':                                    ['iclR', 'kdgR', 'pcaU', 'pobR'],
    'MarR':                                    ['mprA', 'ohrR', 'slyA'],
    'sigma54':                                 ['rpoN'],
    'sigma70/sigma32':                         ['rpoD', 'rpoH'],
    'Biotin protein ligase ':            ['birA'],
    'Class-II aminoacyl-tRNA synthetase':['hisZ'],
    'CsrA':                                    ['csrA'],
    'XRE':                                     ['prtR', 'sutR'],
    'CopG/Arc/MetJ':                           ['copG'],
    'TetR':                                    ['betI'],
    'ArsR':                                    ['arsR2', 'bigR', 'sdpR'],
    'IscR':                                    ['iscR', 'nsrR'],
    'CRP':                                     ['cmr'],
    'YebC':                                    ['yebC'],
    'NtrC/NifA':                               ['norR'],
    'HxlR':                                    ['hxlR', 'yodB', 'ytcD', 'yybR'],
    'RecX':                                    ['recX'],
    'Site-2 Protease (S2P)':                   ['rseP'],
    'Kinase-PPPase':                           ['ppsR'],
    'Fis':                                     ['fis'],
    'Non-ribosomal peptide synthetase':        ['angR'],
    'DnaA':                                    ['hda'],
    'LexA':                                    ['lexA'],
    '(p)ppGpp':                               ['relA'],
}

# MAP REGULATOR TO FAMILY


regulator_to_family = {}

for family, regulators in regulator_classification.items():

    for regulator in regulators:
        regulator_to_family[regulator.lower()] = family

print(f"\nNumber of families: {len(regulator_classification)}")


# 2. LOAD ALL CSV FILES


csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]

print(f"\nFound {len(csv_files)} CSV files")

if not csv_files:
    print("No CSV files found!")
    exit()

all_data = {}
antibiotic_names = []
summary_stats = {}


# PROCESS EACH FILE

# NOTE: each file's 'Regulator' column already contains only
# regulator genes for that antibiotic treatment - every row
# is a regulator, so there is no target-gene distinction to make.
# We only classify each regulator into its family.

for file in csv_files:

    antibiotic_name = (
        file.replace('.csv', '')
        .replace('_regulators', '')
        .replace('_TFs', '')
    )

    antibiotic_names.append(antibiotic_name)

    df = pd.read_csv(file)

    
    # CHECK REQUIRED COLUMN
    

    if 'Regulator' not in df.columns:
        print(f"'Regulator' column not found in {file}")
        continue

    
    # CLEAN REGULATOR COLUMN
    

    df['Regulator'] = (
        df['Regulator']
        .fillna('')
        .astype(str)
        .str.lower()
        .str.strip()
    )

    df = df[df['Regulator'] != '']

    
    # REMOVE COMPLETE DUPLICATES
    

    df = df.drop_duplicates()

    
    # CLASSIFY REGULATOR FAMILY
    

    df['Regulator_Family'] = df['Regulator'].apply(
        lambda gene: regulator_to_family.get(gene, 'Unclassified')
    )

    
    # UNIQUE REGULATORS
    

    unique_regulator_df = df.drop_duplicates(subset=['Regulator'])

    n_regulators = unique_regulator_df['Regulator'].nunique()

    regulator_list = unique_regulator_df['Regulator'].tolist()

    
    # STORE SUMMARY
    

    summary_stats[antibiotic_name] = {

        'unique_regulators': n_regulators,

        'regulator_list': regulator_list
    }

    print(f"{antibiotic_name}: Regulators={n_regulators}")

    all_data[antibiotic_name] = df


# CHECK DATA LOADED


if not all_data:
    print("No valid data loaded!")
    exit()


# FIGURE 1: REGULATOR FAMILY HEATMAP


print("\nCreating Figure 1")

antibiotic_family_counts = {}

for ab, df in all_data.items():

    regulators_only = df.drop_duplicates(subset=['Regulator'])

    antibiotic_family_counts[ab] = (

        regulators_only['Regulator_Family']
        .value_counts()
    )


# BUILD COMPARISON MATRIX


all_families = set()

for counts in antibiotic_family_counts.values():
    all_families.update(counts.index)

comparison_df = pd.DataFrame(

    index=list(all_families),
    columns=antibiotic_names
)

for ab in antibiotic_names:

    if ab in antibiotic_family_counts:

        for family in all_families:

            comparison_df.loc[family, ab] = (
                antibiotic_family_counts[ab].get(family, 0)
            )

comparison_df = comparison_df.fillna(0).astype(int)

families_with_regulators = [

    f for f in comparison_df.index

    if comparison_df.sum(axis=1)[f] > 0
]

heatmap_data = comparison_df.loc[families_with_regulators]


# PLOT FIGURE 1


if len(heatmap_data) > 0:

    fig1, ax1 = plt.subplots(

        figsize=(14, max(8, len(heatmap_data) * 0.4))
    )

    sns.heatmap(

        heatmap_data,

        annot=True,

        fmt='.0f',

        cmap='YlOrRd',

        linewidths=1,

        linecolor='white',

        cbar_kws={

            'label': 'Number of Unique Regulators',
            'shrink': 0.8
        },

        annot_kws={

            'size': 12,
            'weight': 'bold'
        },

        ax=ax1
    )

    ax1.set_xlabel(

        'Antibiotic Treatment',
        fontsize=12,
        fontweight='bold'
    )

    ax1.set_ylabel(

        'Regulator Family',
        fontsize=12,
        fontweight='bold'
    )

    ax1.set_xticklabels(

        ax1.get_xticklabels(),

        rotation=45,
        ha='right',
        fontsize=12
    )

    ax1.set_yticklabels(

        ax1.get_yticklabels(),
        fontsize=12
    )

    ax1.set_title(

        'Regulator Family Distribution Across Antibiotics',

        fontsize=14,
        fontweight='bold',
        pad=15
    )

    plt.tight_layout()

    plt.savefig(
        f'{output_dir}/Figure1_Regulator_Family_Heatmap.png'
    )

    plt.savefig(
        f'{output_dir}/Figure1_Regulator_Family_Heatmap.pdf'
    )

    plt.savefig(
        f'{output_dir}/Figure1_Regulator_Family_Heatmap.svg'
    )

    plt.close()

    print("Saved: Figure1")


# FIGURE 2: UNIQUE REGULATORS PER ANTIBIOTIC


print("\nCreating Figure 2")

regulator_counts = [

    summary_stats[ab]['unique_regulators']

    for ab in antibiotic_names
]

colors_bar = plt.cm.viridis(

    np.linspace(0.2, 0.8, len(antibiotic_names))
)

fig2, ax2 = plt.subplots(figsize=(9, 7))

ax2.bar(

    antibiotic_names,

    regulator_counts,

    color=colors_bar,

    edgecolor='black',

    linewidth=1.5
)

ax2.set_xlabel(

    'Antibiotic Treatment',

    fontsize=12,

    fontweight='bold'
)

ax2.set_ylabel(

    'Unique Regulators',

    fontsize=12,

    fontweight='bold'
)

ax2.set_xticklabels(

    antibiotic_names,

    rotation=45,

    ha='right',

    fontsize=12,

    fontweight='bold'
)

ax2.set_title(

    'Unique Regulators by Antibiotic Treatment',

    fontsize=14,

    fontweight='bold',

    pad=15
)

ax2.set_ylim(0, max(regulator_counts) * 1.15)

plt.tight_layout()

plt.savefig(f'{output_dir}/Figure2_Unique_Regulators.png')
plt.savefig(f'{output_dir}/Figure2_Unique_Regulators.pdf')
plt.savefig(f'{output_dir}/Figure2_Unique_Regulators.svg')

plt.close()

print("Saved: Figure2")


# FIGURE 3: TOP REGULATOR FAMILIES (OVERALL)


print("\nCreating Figure 3")

if len(heatmap_data) > 0:

    overall_family_counts = (

        heatmap_data.sum(axis=1)

        .sort_values(ascending=False)
    )

    top_families = overall_family_counts.head(10)

    fig3, ax3 = plt.subplots(figsize=(9, 7))

    ax3.barh(

        range(len(top_families)),

        top_families.values,

        color=plt.cm.Set3(
            np.linspace(0, 1, len(top_families))
        ),

        edgecolor='black',

        linewidth=1.5
    )

    ax3.set_yticks(range(len(top_families)))

    ax3.set_yticklabels(

        top_families.index,

        fontsize=12,

        fontweight='bold'
    )

    ax3.set_xlabel(

        'Total Unique Regulators',

        fontsize=12,

        fontweight='bold'
    )

    ax3.set_ylabel(

        'Regulator Family',

        fontsize=12,

        fontweight='bold'
    )

    ax3.set_title(

        'Top 10 Regulator Families (Overall)',

        fontsize=14,

        fontweight='bold',

        pad=15
    )

    ax3.invert_yaxis()

    plt.tight_layout()

    plt.savefig(f'{output_dir}/Figure3_Top_Regulator_Families.png')
    plt.savefig(f'{output_dir}/Figure3_Top_Regulator_Families.pdf')
    plt.savefig(f'{output_dir}/Figure3_Top_Regulator_Families.svg')

    plt.close()

    print("Saved: Figure3")

    
    # FIGURE 4: REGULATOR FAMILY SHARE (PIE)
    

    print("\nCreating Figure 4")

    family_counts_plot = overall_family_counts.copy()

    other_threshold = 3

    other_count = (

        family_counts_plot[
            family_counts_plot < other_threshold
        ].sum()
    )

    family_counts_plot = (

        family_counts_plot[
            family_counts_plot >= other_threshold
        ]
    )

    if other_count > 0:

        family_counts_plot = pd.concat([

            family_counts_plot,

            pd.Series({'Other': other_count})
        ])

    colors_pie = plt.cm.Set3(

        np.linspace(0, 1, len(family_counts_plot))
    )

    fig4, ax4 = plt.subplots(figsize=(9, 7))

    wedges, texts, autotexts = ax4.pie(

        family_counts_plot.values,

        labels=family_counts_plot.index,

        autopct='%1.1f%%',

        colors=colors_pie,

        textprops={

            'fontsize': 12,

            'weight': 'bold'
        },

        pctdistance=0.75
    )

    for autotext in autotexts:

        autotext.set_fontsize(12)

        autotext.set_fontweight('bold')

        autotext.set_color('white')

    ax4.set_title(

        'Regulator Family Share (Overall)',

        fontsize=14,

        fontweight='bold',

        pad=15
    )

    ax4.set_aspect('equal')

    plt.tight_layout()

    plt.savefig(f'{output_dir}/Figure4_Regulator_Family_Share.png')
    plt.savefig(f'{output_dir}/Figure4_Regulator_Family_Share.pdf')
    plt.savefig(f'{output_dir}/Figure4_Regulator_Family_Share.svg')

    plt.close()

    print("Saved: Figure4")


# EXPORT TABLES


summary_df = pd.DataFrame([{

    'Antibiotic': ab,

    'Unique_Regulators': stats['unique_regulators']

} for ab, stats in summary_stats.items()])

summary_df.to_csv(

    f'{output_dir}/summary_statistics.csv',

    index=False
)

if len(heatmap_data) > 0:

    heatmap_data.to_csv(

        f'{output_dir}/regulator_family_comparison_matrix.csv'
    )


# EXPORT PER-ANTIBIOTIC CSV FILES WITH REGULATOR_FAMILY COLUMN

# One CSV per antibiotic treatment, containing the original
# 'Regulator' column plus the newly assigned 'Regulator_Family'
# column for every regulator gene.

print("\nExporting per-antibiotic CSV files with Regulator_Family column")

for ab, df in all_data.items():

    out_df = df.drop_duplicates(subset=['Regulator']).copy()

    out_path = f'{output_dir}/{ab}_regulator_family.csv'

    out_df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print(f"\nOutputs saved in: {output_dir}/")

print("\nCounts are based on UNIQUE regulator genes only")

REGULATOR FAMILY ANALYSIS ACROSS ANTIBIOTIC TREATMENTS

Number of families: 42

Found 7 CSV files
Amikacin: Regulators=27
Ciprofloxacin: Regulators=104
Colistin Resistance: Regulators=123
Doripenem: Regulators=94
Meropenem: Regulators=42
Polymyxin: Regulators=46
Tigecycline: Regulators=96

Creating Figure 1
Saved: Figure1

Creating Figure 2
Saved: Figure2

Creating Figure 3
Saved: Figure3

Creating Figure 4
Saved: Figure4

Exporting per-antibiotic CSV files with Regulator_Family column
Saved: publication_figures/Amikacin_regulator_family.csv
Saved: publication_figures/Ciprofloxacin_regulator_family.csv
Saved: publication_figures/Colistin Resistance_regulator_family.csv
Saved: publication_figures/Doripenem_regulator_family.csv
Saved: publication_figures/Meropenem_regulator_family.csv
Saved: publication_figures/Polymyxin_regulator_family.csv
Saved: publication_figures/Tigecycline_regulator_family.csv

ANALYSIS COMPLETE

Outputs saved in: publication_figures/

Counts are based on UNIQUE r